# Module 12 Lab: Prescriptive Analytics

## Objective

In this lab, you will decide which students to prioritise under real-world constraints.

By the end of this lab, you should be able to:

- move from prediction to decision
- apply simple prioritisation rules
- account for budget constraints
- compare heuristic and optimisation-based choices

## Why this matters

A prediction tells us what may happen.

But in practice, we still need to decide:

- who to act on
- how to allocate limited resources
- what trade-offs are acceptable

Prescriptive analytics helps us move from:

**prediction → decision → action**

## Scenario

You are given a list of students with predicted risk scores.

You also know that each intervention has a cost.

Your team has a limited budget.

Your task is to decide:

- which students to prioritise
- how to make the best use of limited resources

In [ ]:
import pandas as pd
from IPython.display import display
from itertools import combinations

## Step 1: Load the data

Each row contains:

- a student
- a predicted risk score
- an estimated intervention cost

Look at the data first and think about:

- Who has the highest risk?
- Which students are cheaper or more expensive to support?
- What trade-offs might appear?

In [ ]:
df = pd.DataFrame({
    "student": list("ABCDEFGHIJ"),
    "risk_score": [0.90, 0.85, 0.80, 0.70, 0.60, 0.55, 0.50, 0.45, 0.40, 0.30],
    "intervention_cost": [50, 60, 40, 30, 20, 25, 15, 10, 10, 5]
})

display(df)

## Guided interpretation

Reflect on the data:

- Which students appear highest priority if you look only at risk?
- Which students may be expensive to support?
- Do high risk and high cost always go together?

Write your observations in a new markdown cell.

## Step 2: Baseline approach — rank by risk

A simple starting point is to rank students by risk score.

This assumes:

- higher predicted risk = higher priority

In [ ]:
df_sorted = df.sort_values(by="risk_score", ascending=False)
display(df_sorted)

## Guided interpretation

This ranking shows who appears most at risk.

But ask yourself:

- Is this enough to make a decision?
- Does this consider limited resources?

## Step 3: Naive selection — choose the top 3 students

Let us first select the top 3 students by risk score.

In [ ]:
top3 = df_sorted.head(3)
display(top3)

print("Total cost:", top3["intervention_cost"].sum())
print("Total risk covered:", round(top3["risk_score"].sum(), 2))

## Guided interpretation

Think about:

- Which students were selected?
- Is the total cost acceptable?
- Would this still work if the budget were limited?

Write your thoughts in a new markdown cell.

## Step 4: Add a budget constraint

In reality, resources are limited.

Let us assume the total budget available is:

**100**

In [ ]:
budget = 100
budget

## Step 5: Select by highest risk, but stay within budget

We now go down the risk-ranked list and select students only if their intervention cost fits within the budget.

In [ ]:
selected = []
total_cost = 0
total_risk = 0

for _, row in df_sorted.iterrows():
    if total_cost + row["intervention_cost"] <= budget:
        selected.append(row["student"])
        total_cost += row["intervention_cost"]
        total_risk += row["risk_score"]

print("Selected students:", selected)
print("Total cost:", total_cost)
print("Total risk covered:", round(total_risk, 2))

## Guided interpretation

Now reflect:

- How did the selection change once the budget was introduced?
- Did the highest-risk students always get selected?
- What trade-off are you making?

Write your observations in a new markdown cell.

## Step 6: Consider value — risk per unit cost

Instead of looking only at risk, we can also consider value.

A simple value score is:

**risk score / intervention cost**

This helps us identify students who offer more expected value per unit of cost.

In [ ]:
df["value_score"] = df["risk_score"] / df["intervention_cost"]
display(df.sort_values(by="value_score", ascending=False))

## Guided interpretation

Think about:

- Which students now appear more attractive?
- Why might lower-cost students move up in priority?
- Is “highest risk” always the same as “best choice”?

Write a short note in a new markdown cell.

## Step 7: Select by value score within budget

Now let us prioritise students using the value score instead of risk alone.

In [ ]:
df_value_sorted = df.sort_values(by="value_score", ascending=False)

selected_value = []
total_cost_value = 0
total_risk_value = 0

for _, row in df_value_sorted.iterrows():
    if total_cost_value + row["intervention_cost"] <= budget:
        selected_value.append(row["student"])
        total_cost_value += row["intervention_cost"]
        total_risk_value += row["risk_score"]

print("Selected students:", selected_value)
print("Total cost:", total_cost_value)
print("Total risk covered:", round(total_risk_value, 2))

## Guided interpretation

Compare this with the previous approach:

- Did the selected students change?
- Is this more efficient?
- What are the advantages and disadvantages of this rule?

Write your reflections in a new markdown cell.

## Step 8: Compare the two decision rules

Let us compare:

- risk-first selection
- value-based selection

In [ ]:
comparison = pd.DataFrame({
    "Approach": ["Risk-first", "Value-based"],
    "Selected Students": [", ".join(selected), ", ".join(selected_value)],
    "Total Cost": [total_cost, total_cost_value],
    "Total Risk Covered": [round(total_risk, 2), round(total_risk_value, 2)]
})

display(comparison)

## Guided interpretation

Think about:

- Which rule covered more total risk?
- Which rule made better use of budget?
- Which would you use in practice, and why?

Write your answer in a new markdown cell.

## Step 9: (Optional) Optimisation approach

So far, we used simple decision rules.

Now we try a more formal approach:

- evaluate all possible student combinations
- choose the combination that gives the highest total risk score
- stay within budget

This is a simple form of optimisation.

In [ ]:
best_score = 0
best_cost = 0
best_combo = None

students = df.to_dict("records")

for r in range(1, len(students) + 1):
    for combo in combinations(students, r):
        total_combo_cost = sum(s["intervention_cost"] for s in combo)
        total_combo_score = sum(s["risk_score"] for s in combo)

        if total_combo_cost <= budget and total_combo_score > best_score:
            best_score = total_combo_score
            best_cost = total_combo_cost
            best_combo = combo

best_students = [s["student"] for s in best_combo]

print("Best student set:", best_students)
print("Total cost:", best_cost)
print("Total risk covered:", round(best_score, 2))

## Guided interpretation

Now compare optimisation with the earlier heuristics:

- Did the result improve?
- Was the value-based rule close to the best solution?
- When might a simple heuristic be enough?

Write your observations in a new markdown cell.

## Step 10: Think about real-world complexity

In practice, decisions are often more complex.

You may also need to consider:

- likelihood that an intervention will work
- fairness across student groups
- different intervention types
- limited staff time
- urgency of each case

This means real prescriptive analytics often goes beyond a simple score.

## Final reflection

Write a short summary covering:

### 1. Decision strategy
What rule did you use?

### 2. Trade-offs
What did you prioritise?

### 3. Constraints
How did the budget affect decisions?

### 4. Comparison
How did the heuristic and optimisation approaches differ?

### 5. Real-world application
How would this scale to a real analytics system?

## Key takeaway

Predictions do not create value on their own.

Prescriptive analytics helps us decide:

- who to act on
- how to allocate limited resources
- what action may lead to the greatest impact